# Tutorial 7 — Clustering and Metric Geometry

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 7: Unsupervised Learning — Clustering**

---

Everything so far has been supervised: we produced labels and asked a model to
reproduce them. Now the labels go away. Given only a point cloud, what can be
recovered?

Lecture 7 covered three answers, and the interesting thing for a geometer is that
they encode three *different* ideas of what a cluster is:

| method | a cluster is… | the geometry it sees |
|---|---|---|
| **k-means** | points near a common centre | Euclidean, convex cells |
| **spectral** | a region weakly connected to the rest | the graph Laplacian |
| **DBSCAN** | a maximal chain of dense points | local density, any shape |

We will make the differences bite by asking a question with a known answer: **how
many connected components does a real algebraic curve have?**

| § | Question |
|---|---|
| 1 | Point clouds from manifolds and algebraic sets |
| 2 | k-means, and what "convex cell" costs you |
| 3 | Spectral clustering and the graph Laplacian |
| 4 | DBSCAN, noise, and detecting a geometric phase transition |
| 5 | Summary |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

SEED = 20260915
rng = np.random.default_rng(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("numpy", np.__version__)

---
## 1. Three point clouds with known answers

We build three datasets whose correct clustering we know in advance, chosen so that
the three algorithms disagree about them.

- **Blobs** — three Gaussian lumps. Every method should succeed; this is the
  control.
- **Concentric circles** — two components, neither of which is convex, and whose
  *centroids coincide*. This is designed to defeat k-means.
- **An elliptic curve** $y^2 = x^3 + px + q$ — a genuine algebraic set. Over
  $\mathbb{R}$ it has **two** connected components when the cubic has three real
  roots, and **one** otherwise; the transition happens exactly at the discriminant
  $\Delta = -16(4p^3 + 27q^2) = 0$. That gives us a geometric phase with an exactly
  known boundary, which §4 will try to recover from samples alone.

In [ ]:
def make_blobs(n, rng):
    centres = np.array([[0.0, 0.0], [3.0, 0.5], [1.4, 2.8]])
    lab = rng.integers(0, 3, n)
    return centres[lab] + 0.42 * rng.normal(size=(n, 2)), lab


def make_circles(n, rng):
    lab = rng.integers(0, 2, n)
    r = np.where(lab == 0, 1.0, 2.4) + 0.10 * rng.normal(size=n)
    t = rng.uniform(0, 2 * np.pi, n)
    return np.stack([r * np.cos(t), r * np.sin(t)], axis=1), lab


def sample_elliptic(n, p, q, rng, noise=0.02, x_max=3.0):
    """Points on y^2 = x^3 + p x + q, by rejection on the region where it is >= 0."""
    pts = []
    while len(pts) < n:
        x = rng.uniform(-x_max, x_max, 4 * n)
        rhs = x**3 + p * x + q
        x = x[rhs >= 0]
        y = np.sqrt(x**3 + p * x + q) * rng.choice([-1.0, 1.0], len(x))
        pts.append(np.stack([x, y], axis=1))
        if sum(len(a) for a in pts) == 0:
            raise ValueError("empty real locus")
    P = np.concatenate(pts)[:n]
    return P + noise * rng.normal(size=P.shape)


def n_components_exact(p, q):
    """Real components of y^2 = x^3+px+q: 2 if the cubic has 3 real roots, else 1."""
    return 2 if (4 * p**3 + 27 * q**2) < 0 else 1


X_blob, y_blob = make_blobs(600, rng)
X_circ, y_circ = make_circles(600, rng)
X_ell = sample_elliptic(800, -3.0, 1.0, rng, x_max=2.5)

print(f"elliptic curve p=-3, q=1: discriminant term 4p^3+27q^2 = {4*(-3)**3 + 27*1**2}"
      f"  ->  {n_components_exact(-3.0, 1.0)} real components")

fig, axes = plt.subplots(1, 3, figsize=(10.6, 3.0))
for ax, (X, name) in zip(axes, [(X_blob, "three blobs"), (X_circ, "concentric circles"),
                                (X_ell, "$y^2 = x^3 - 3x + 1$")]):
    ax.scatter(*X.T, s=5, color=GEO_DARK, alpha=0.6)
    ax.set_aspect("equal"); ax.set_title(name)
plt.tight_layout(); plt.show()

---
## 2. k-means: clusters as Voronoi cells

Lloyd's algorithm alternates two steps — assign each point to the nearest centre,
then move each centre to the mean of its points. It is coordinate descent on

$$\Phi(\{\mu_c\}) \;=\; \sum_i \min_c \lVert x_i - \mu_c\rVert^2 ,$$

so the objective decreases monotonically and it converges. What it converges *to* is
determined by initialisation — the objective is not convex.

The geometry is fixed by the assignment step: each cluster is the intersection of
the data with a **Voronoi cell** of the centres, so it is convex. That is not a
tuning issue you can fix with more iterations. It is what the algorithm means by
"cluster".

In [ ]:
def kmeans(X, k, rng, iters=100):
    """Lloyd's algorithm with k-means++ initialisation."""
    centres = [X[rng.integers(len(X))]]
    for _ in range(k - 1):                       # k-means++ seeding
        d2 = np.min(((X[:, None, :] - np.array(centres)[None])**2).sum(-1), axis=1)
        centres.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    C = np.array(centres)
    for _ in range(iters):
        lab = np.argmin(((X[:, None, :] - C[None])**2).sum(-1), axis=1)
        newC = np.array([X[lab == c].mean(0) if (lab == c).any() else C[c]
                         for c in range(k)])
        if np.allclose(newC, C):
            break
        C = newC
    inertia = ((X - C[lab])**2).sum()
    return lab, C, inertia


lab_b, C_b, _ = kmeans(X_blob, 3, rng)
lab_c, C_c, _ = kmeans(X_circ, 2, rng)

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4))
for ax, (X, lab, C, name) in zip(axes, [(X_blob, lab_b, C_b, "blobs: correct"),
                                        (X_circ, lab_c, C_c, "circles: hopeless")]):
    ax.scatter(*X.T, c=lab, cmap="coolwarm", s=6)
    ax.scatter(*C.T, marker="X", s=150, color="k", zorder=5)
    ax.set_aspect("equal"); ax.set_title(name)
plt.tight_layout(); plt.show()

from scipy.optimize import linear_sum_assignment

def acc(pred, true):
    """Clustering accuracy under the best one-to-one relabeling (any number of clusters)."""
    lp, lt = np.unique(pred), np.unique(true)
    cost = np.array([[-np.sum((pred == a) & (true == b)) for b in lt] for a in lp])
    row, col = linear_sum_assignment(cost)
    return -cost[row, col].sum() / len(true)

print(f"blobs   agreement with truth: {acc(lab_b, y_blob):.1%}   (chance is {1/3:.0%})")
print(f"circles agreement with truth: {acc(lab_c, y_circ):.1%}   (chance is 50%)")
print(f"\ncentroids of the two true circles:")
for c in (0, 1):
    print(f"   component {c}: {X_circ[y_circ == c].mean(0).round(3)}")
print('...approximately equal so k-means cannot separate them!')

The failure on the circles is total and completely predictable: the two true
components have **the same centroid**, so no assignment of points to centres can
separate them. k-means is not doing badly here — it is answering a different
question correctly.

The lesson generalises past this toy. Before reaching for a clustering method, ask
what shape of cluster it presumes. If your components are elongated, nested,
curved, or of very different densities — which is normal for geometric data —
centroid-based methods are the wrong tool by construction.

> **Exercise 1 — the geometry of $k$-means.**
> (a) The objective decreases monotonically but the result depends on
> initialisation. Run k-means on the blobs from 50 random seeds and histogram the
> final inertia. How often does it find the best solution, and does k-means++
> seeding help?
>
> (b) Plot inertia against $k$ for the blobs (the "elbow method"). Now do the same
> for the circles. What does the elbow tell you when the model is wrong?
>
> (c) Apply k-means to the elliptic curve with $k=2$. Does it recover the two real
> components? Explain the result from the Voronoi picture before you run it.

---
## 3. Spectral clustering: connectivity instead of proximity

Spectral clustering replaces "near a common centre" by "**connected**". Build a
similarity graph — here $k$-nearest-neighbours — form the graph Laplacian
$L = D - W$, and use its low eigenvectors as coordinates.

The fact that does the work is one you already know from Tutorial 3 §3:

> the multiplicity of the eigenvalue $0$ of $L$ equals the number of connected
> components of the graph, and the corresponding eigenvectors are constant on each
> component.

So the number of clusters is not a parameter to be guessed — it can be **read off
the spectrum**. That is $b_0 = \dim\ker\Delta$ in Hodge theory, discretised.

In practice the eigenvalues are never exactly zero, because a finite sample of a
connected component is a connected graph with a small but positive Fiedler value.
So one looks for the **spectral gap** — the largest jump in the sorted spectrum —
rather than applying an absolute threshold. Where you place that cut is the real
choice, and the next cell shows it being easy in two cases and genuinely hard in
the third.

In [ ]:
def knn_graph(X, k=8):
    """Symmetric k-NN adjacency with unit weights."""
    tree = cKDTree(X)
    _, idx = tree.query(X, k=k + 1)
    n = len(X)
    W = np.zeros((n, n))
    rows = np.repeat(np.arange(n), k)
    W[rows, idx[:, 1:].ravel()] = 1.0
    return np.maximum(W, W.T)


def laplacian_spectrum(W, n_eig=8):
    d = W.sum(1)
    Dm = 1 / np.sqrt(np.maximum(d, 1e-12))
    L = np.eye(len(W)) - (Dm[:, None] * W) * Dm[None, :]      # normalised Laplacian
    vals, vecs = np.linalg.eigh(L)
    return vals[:n_eig], vecs[:, :n_eig]


def spectral_cluster(X, k_clusters, rng, k_nn=8):
    W = knn_graph(X, k_nn)
    vals, vecs = laplacian_spectrum(W, k_clusters)
    U = vecs / np.maximum(np.linalg.norm(vecs, axis=1, keepdims=True), 1e-12)
    lab, _, _ = kmeans(U, k_clusters, rng)
    return lab, vals


def n_components_from_gap(vals):
    """Count near-zero eigenvalues by the largest gap, not an absolute threshold."""
    return int(np.argmax(np.diff(vals))) + 1


for name, X in [("blobs", X_blob), ("circles", X_circ), ("elliptic", X_ell)]:
    vals, _ = laplacian_spectrum(knn_graph(X, 8), 6)
    print(f"{name:9s} " + " ".join(f"{v:7.4f}" for v in vals)
          + f"   largest gap after index {n_components_from_gap(vals) - 1}"
          + f"  ->  {n_components_from_gap(vals)} components")

In [ ]:
lab_sb, _ = spectral_cluster(X_blob, 3, rng)
lab_sc, _ = spectral_cluster(X_circ, 2, rng)
lab_se, _ = spectral_cluster(X_ell, 2, rng)

fig = plt.figure(figsize=(11.5, 6.4))
gs = fig.add_gridspec(2, 6)

ax0 = fig.add_subplot(gs[0, 0:2])
ax0.scatter(*X_blob.T, c=lab_sb, cmap="coolwarm", s=6)
ax0.set_aspect("equal"); ax0.set_title(f"blobs: {acc(lab_sb, y_blob):.0%} correct")

ax1 = fig.add_subplot(gs[0, 2:4])
ax1.scatter(*X_circ.T, c=lab_sc, cmap="coolwarm", s=6)
ax1.set_aspect("equal"); ax1.set_title(f"circles: {acc(lab_sc, y_circ):.0%} correct")

ax2 = fig.add_subplot(gs[0, 4:6])
ax2.scatter(*X_ell.T, c=lab_se, cmap="coolwarm", s=6)
ax2.set_aspect("equal"); ax2.set_title("elliptic curve: two components")

vals_b, _ = laplacian_spectrum(knn_graph(X_blob, 8), 8)
vals_c, _ = laplacian_spectrum(knn_graph(X_circ, 8), 8)
vals_e, _ = laplacian_spectrum(knn_graph(X_ell, 8), 8)
eps = 1e-10   # eigenvalues can be ~0 (or slightly negative from roundoff); clip for the log axis

ax_lin = fig.add_subplot(gs[1, 0:3])
ax_lin.plot(vals_b, "o-", color=GEO_TEAL, label="blobs")
ax_lin.plot(vals_c, "s-", color=GEO_RUST, label="circles")
ax_lin.plot(vals_e, "^-", color=GEO_DARK, label="elliptic")
ax_lin.axhline(0, color="k", lw=0.8)
ax_lin.set_xlabel("index"); ax_lin.set_ylabel("eigenvalue of $L$")
ax_lin.set_title("linear scale"); ax_lin.legend(fontsize=8)


ax_log = fig.add_subplot(gs[1, 3:6])
ax_log.plot(np.clip(vals_b, eps, None), "o-", color=GEO_TEAL, label="blobs")
ax_log.plot(np.clip(vals_c, eps, None), "s-", color=GEO_RUST, label="circles")
ax_log.plot(np.clip(vals_e, eps, None), "^-", color=GEO_DARK, label="elliptic")
ax_log.set_yscale("log")
ax_log.set_xlabel("index"); ax_log.set_ylabel("eigenvalue of $L$ (log scale)")
ax_log.set_title("log scale"); ax_log.legend(fontsize=8)
plt.tight_layout(); plt.show()

Read the three lines of output carefully, because they do not all say the same
thing.

For the **blobs** and the **circles** the gap is unmistakable: three (respectively
two) tiny eigenvalues, then a jump of an order of magnitude. The count is
unambiguous and correct.

For the **elliptic curve** it is not. The two components are cleanly separated in
space, yet the spectrum has no clean gap — because each component is a long, thin
curve, and a long thin graph has a small Fiedler value of its *own*. Cutting a
curve in half is nearly as cheap as separating the two branches, so the "number of
near-zero eigenvalues" stops being well defined. This is a real and recurring
limitation: spectral counting works well for fat, blobby components and poorly for
filamentary ones, which is exactly the shape most algebraic curves have.

Given the number of clusters, spectral clustering still *separates* both the circles
and the two branches correctly — connectivity is the right notion, and the Laplacian
sees it. It is the **counting** that fails on thin sets, and that is what motivates
the density-based method in §4.

A note on the two panels above: the linear-scale plot is what the gap-counting
argument actually relies on — an honest additive difference between consecutive
eigenvalues. The log-scale plot compresses every genuinely small eigenvalue into
the same visual neighbourhood while blowing up the transition from the exact-zero
eigenvalues (clipped to $10^{-10}$) to the first positive one into a jump of many
orders of magnitude, for *every* dataset alike. That makes the log panel a poor
tool for judging cluster count by eye — use it only to see the overall spread of
the spectrum, not to locate the gap that matters.

Note what changed: not the data, and not the amount of computation, but **the
metric structure we chose to impose**. The $k$-NN graph declares which points are
neighbours; everything downstream is linear algebra on that choice. Geometric
judgement enters at the graph-building step, and that is where you should spend your
attention.

> **Exercise 2 — the graph is the model.**
> (a) Sweep $k$ in the $k$-NN graph from 3 to 40 on the concentric circles and plot
> the number of near-zero eigenvalues. For which $k$ do the two circles merge into
> one component, and can you predict that value from the gap between the circles and
> the sampling density?
>
> (b) Replace the $k$-NN graph by a Gaussian kernel $W_{ij} = e^{-\lVert x_i-x_j\rVert^2/2t}$
> and sweep $t$. Which parametrisation is more robust here?
>
> (c) The **Fiedler vector** (the eigenvector of the smallest non-zero eigenvalue)
> gives a bipartition even when the graph is connected. Plot it as a colour on the
> elliptic curve for a case with only *one* real component. What does it split, and
> is the split geometrically meaningful?

---
## 4. DBSCAN, and detecting a geometric phase transition

DBSCAN takes a third view: a cluster is a maximal set of points connected by hops of
length $\le \varepsilon$ through regions of sufficient density. Points in sparse
regions are labelled **noise** and belong to no cluster — the only one of the three
methods that may refuse to classify a point.

For our purposes its useful property is that it determines the number of clusters
itself. That lets us do something the other two cannot: sweep a *parameter of the
geometry* and detect where the topology changes.

The elliptic curve family $y^2 = x^3 + px + q$ has two real components exactly when
$4p^3 + 27q^2 < 0$. We fix $p = -3$, vary $q$, and ask DBSCAN — which knows no
algebra whatever — to count components from samples. The transition should appear at
$q^2 = -4p^3/27 = 4$, i.e. at $q = \pm 2$.

In [ ]:
def dbscan(X, eps, min_pts=5):
    """DBSCAN. Labels are 0,1,2,...; -1 means noise."""
    tree = cKDTree(X)
    neigh = tree.query_ball_point(X, eps)
    core = np.array([len(nb) >= min_pts for nb in neigh])
    labels = np.full(len(X), -1)
    cid = 0
    for i in range(len(X)):
        if labels[i] != -1 or not core[i]:
            continue
        stack, labels[i] = [i], cid          # flood fill from a core point
        while stack:
            j = stack.pop()
            for m in neigh[j]:
                if labels[m] == -1:
                    labels[m] = cid
                    if core[m]:
                        stack.append(m)
        cid += 1
    return labels


lab_db = dbscan(X_ell, eps=0.6, min_pts=4)
print(f"elliptic curve (p=-3, q=1): DBSCAN finds "
      f"{len(set(lab_db[lab_db >= 0]))} clusters, {(lab_db < 0).sum()} noise points"
      f"   (truth: {n_components_exact(-3.0, 1.0)})")

In [ ]:
P_FIX = -3.0
qs = np.linspace(-3.2, 3.2, 33)
found, truth = [], []
for q in qs:
    try:
        Xq = sample_elliptic(800, P_FIX, q, rng, noise=0.015, x_max=2.5)
        lab = dbscan(Xq, eps=0.6, min_pts=4)
        found.append(len(set(lab[lab >= 0])))
    except Exception:
        found.append(np.nan)
    truth.append(n_components_exact(P_FIX, q))
found, truth = np.array(found, float), np.array(truth)

q_star = np.sqrt(-4 * P_FIX**3 / 27)
agree = np.mean(found == truth)
print(f"predicted transition at |q| = {q_star:.4f}")
print(f"DBSCAN agrees with the exact component count on {agree:.0%} of the sweep")

fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.3))
axes[0].step(qs, truth, where="mid", color=GEO_DARK, lw=2, label="exact")
axes[0].plot(qs, found, "o", color=GEO_RUST, ms=5, label="DBSCAN")
for s in (-1, 1):
    axes[0].axvline(s * q_star, color=GEO_TEAL, ls="--", lw=1.3)
axes[0].set_xlabel("$q$   (with $p = -3$)"); axes[0].set_ylabel("components")
axes[0].set_yticks([1, 2]); axes[0].legend(fontsize=8)
axes[0].set_title("a topological phase transition, found from samples")

for ax_q, col in [(1.0, GEO_TEAL), (2.6, GEO_RUST)]:
    Xs = sample_elliptic(500, P_FIX, ax_q, rng, noise=0.015, x_max=2.5)
    axes[1].scatter(*Xs.T, s=4, color=col, alpha=0.6,
                    label=f"$q = {ax_q}$  ({n_components_exact(P_FIX, ax_q)} comp.)")
axes[1].set_aspect("equal"); axes[1].legend(fontsize=8)
axes[1].set_title("either side of the discriminant")
plt.tight_layout(); plt.show()

The step in the exact count sits at $|q| = 2$, where the discriminant vanishes, and
the clustering reproduces it from samples with no knowledge of the equation. This is
a small but genuine instance of the course's thesis: an unsupervised method,
applied to sampled points, recovered a statement about the *topology* of an
algebraic set, and we could check the answer because the discriminant is known.

Two cautions, both of which matter more than the success.

- **Near the transition it is genuinely ambiguous.** Just inside $|q| = 2$ the two
  components exist but are nearly touching, so whether DBSCAN separates them depends
  on $\varepsilon$ and on the sampling density. Any disagreement in the plot will sit
  there. That is not a defect of the method — the sampled data really does not
  distinguish "just connected" from "just disconnected".
- **DBSCAN chose the answer, but we chose $\varepsilon$.** The number of clusters is
  not a free lunch; the scale parameter carries the same information, moved
  somewhere less visible. Exercise 3 makes that explicit.

> **Exercise 3 — how much did $\varepsilon$ decide?**
> (a) Redo the sweep for $\varepsilon \in \{0.3, 0.45, 0.6, 0.9\}$ and plot the
> recovered transition point against $\varepsilon$. How stable is the answer?
>
> (b) Persistent homology answers the same question without a scale choice, by
> tracking components across *all* $\varepsilon$ at once. Compute the number of
> $H_0$ bars that survive a long interval and compare. (This is Tutorial 11's
> subject; a rough version is a few lines with a minimum spanning tree.)
>
> (c) Sweep both $p$ and $q$ on a grid and colour the plane by the recovered
> component count. You should be drawing the discriminant curve
> $4p^3 + 27q^2 = 0$ — a cusp. How much sampling does it take to see the cusp
> clearly?

---
## 5. What to take away

- **The three algorithms encode three different definitions of "cluster"** —
  proximity to a centre, connectivity in a graph, and density. Choosing one is a
  modelling decision about the geometry you expect, not a matter of accuracy.
- **k-means presumes convex clusters.** On the concentric circles the true
  components share a centroid, so no centre-based method can work. Diagnose this
  from the shape of your data, not from a bad score.
- **Spectral clustering counts components with the Laplacian**, and the number of
  clusters can be read off the spectrum rather than guessed —
  $\#\{\lambda \approx 0\} = b_0$, the discrete shadow of $\dim\ker\Delta$.
- **The similarity graph is the real model.** Everything after it is linear
  algebra; all the geometric judgement lives in how you decide who is a neighbour.
- **Unsupervised methods can recover exact mathematics.** DBSCAN found the
  discriminant of an elliptic curve family from samples — and, just as importantly,
  was ambiguous exactly where the mathematics is degenerate.

### Next

**Lecture 8** keeps the unsupervised setting but asks for coordinates rather than
labels: PCA, kernel PCA and nonlinear embeddings. **Tutorial 8** applies them to
manifolds where intrinsic and extrinsic geometry disagree.

### Further reading

- von Luxburg, "A tutorial on spectral clustering", *Statistics and Computing* **17** (2007) — the definitive account of §3.
- Ester et al., "A density-based algorithm for discovering clusters", *KDD* 1996 — the original DBSCAN.
- Silverman, *The Arithmetic of Elliptic Curves*, ch. III — the discriminant and the real locus.
- Chung, *Spectral Graph Theory* — the Laplacian facts used here, in full.